In [1]:
import os
import numpy as np

from typing import Tuple, Dict, Any, Optional

import jax
import jax.numpy as jnp

import flax.linen as nn
import optax

from functools import partial

from tqdm.auto import tqdm

from load_beamng_data import (
    format_dataset,
    exponential_moving_average
)

from engine_model import (
    EngineModel,
    create_networks,
    NoiseTerm,
    EngineModelEnv,
    EngineModelV2,
    ModelWithTargetWheelSpeedAndRate,
    load_model_from_config,
    integrate_path
)

from logging_utils import TrainCheckpoints, load_saved_data_from_checkpoint

from data_utils import (
    split_dataset,
    pick_batch_transitions_as_array,
    remove_irrelevant_fields,
    get_mean_and_scale_fields,
    get_min_max_fields
)
TRAINING_LOGS_DIR = os.getcwd() + "/training_files/"

In [2]:
# Load the dataset used for training
def criteria_valid_traj(data : Dict[str, np.array]):
    # First the vehicle must be in realistic drive mode (as the mode for control)
    is_realistic = data["driveStatus_isRealisticDrive"] == 1
    # For now let's assume 2WD and ig range mode (0)
    valid = (data["brake"] <= 0.001) & \
            (data["pbrake"] <= 0.0001) & \
            (data["driveStatus_mode4WD"] == 0) & \
            (data["driveStatus_modeRangeBox"] == 0) & \
            (data["rear_wheelspeed"] >= 10) & \
            (data["driveStatus_engineRunning"] == 1) & \
            is_realistic
    # valid  = valid & (np.logical_and(data["time"] >141, data["time"] < 155))
    return valid

def extra_processing(data : Dict[str, np.array]):
    # Let's smooth some of the data
    data["rear_wheelspeed"] = exponential_moving_average(data["rear_wheelspeed"], alpha = 0.25)
    data["engine_speed"] = exponential_moving_average(data["engine_speed"], alpha = 0.8)
    data["boost_pressure"] = exponential_moving_average(data["boost_pressure"], alpha = 0.2)
    # Let's compute angular acceleration
    dt = np.mean(np.diff(data["time"]))
    rear_wr_dot = np.gradient(data["rear_wheelspeed"], dt)
    data["rear_wheelspeed_dot"] = exponential_moving_average(rear_wr_dot, alpha = 0.05)
    rear_engine_dot = np.gradient(data["engine_speed"], dt)
    data["engine_speed_dot"] = exponential_moving_average(rear_engine_dot, alpha = 0.2)
    # Let's also add some inverse gear ratio
    data["inv_gear_ratio"] = 1.0 / (8.516 *data["gearRatio"]) # 8.516 ~ final drive ratio
    data["engine_torque"] = data["engineTorque"]

# Let's pick a dataset
m_runs =[9,]
full_dataset = format_dataset(
    m_runs,
    sensor_type = "gtstate",
    vehicle_model = "utv_wild",
    valid_trajectory_cond = criteria_valid_traj,
    extra_processing = extra_processing,
    min_traj_len = 200,
)


========== Loading Run 9 ==========

Processing ('ego', 'gtstate')....
Total number of successful logs: 1
Total number of failed logs: 0
Total number of trajectories: 1

Successful loading : 
-  Run 9


In [3]:
# Get min and max values
min_vals, max_vals = get_min_max_fields(full_dataset)
# Get scale and mean values
mean_vals, scale_vals = get_mean_and_scale_fields(full_dataset)

In [4]:
for name in max_vals.keys():
    print(f"{name}: \t[  {min_vals[name]} | {max_vals[name]}  ]")

pbrake: 	[  0.0 | 0.0  ]
gearRatio: 	[  1.0 | 3.5  ]
gearboxTorque: 	[  -168.70067005305 | 379.43944003966  ]
engineLoad: 	[  0.0 | 1.0  ]
brakeInput: 	[  0.0 | 0.0  ]
engineTorque: 	[  -24.443037122175 | 193.53355930915  ]
wheelRL_brakeTorque: 	[  -0.00017694807052626008 | 0.0002655496597289101  ]
wheelRL_angle: 	[  0.0 | 0.01547566614180664  ]
wheelRL_angVel: 	[  9.3377170562744 | 98.346588134766  ]
wheelRL_angVelB: 	[  9.356894493103 | 99.028747558594  ]
wheelRL_propTorque: 	[  -872.38178718913 | 1408.9160840261  ]
wheelRL_speed: 	[  3.5815783294658 | 38.067495256317  ]
wheelFL_brakeTorque: 	[  -0.00065019083023077 | 0.00061357116699212  ]
wheelFL_angle: 	[  0.020467439202268022 | 0.026931638577914436  ]
wheelFL_angVel: 	[  9.0947351455688 | 97.894386291504  ]
wheelFL_angVelB: 	[  9.1107654571533 | 98.349914550781  ]
wheelFL_propTorque: 	[  -8.3738322485422 | 4.1878871180111  ]
wheelFL_speed: 	[  3.5378448761147 | 38.413822528785  ]
RPM: 	[  2752.3961894332 | 8210.2227127426  ]
acce

In [5]:
# Let's load existing model
# Load the model
ckpt_name = "test_10"
ckpt_cfg, path_ckpt = load_saved_data_from_checkpoint(
    TRAINING_LOGS_DIR, ckpt_name, step = -2
)

Checkpoints directory: /home/franckd/ros2_ws/src/bng_xal/bng_simulator/examples/low_level_model/training_files/test_10/checkpoints
Tensorboard and config directory: /home/franckd/ros2_ws/src/bng_xal/bng_simulator/examples/low_level_model/training_files/test_10
Max to keep: None
Async exec: False
Timeout async: 60
Saving Metrics:
 {'Test/mse': 2.0, 'Train/mse': 1.0}


/home/franckd/.local/lib/python3.10/site-packages/orbax/checkpoint/_src/serialization/type_handlers.py:1175: UserWarning: Couldn't find sharding info under RestoreArgs. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file instead of directly from RestoreArgs. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


In [ ]:
model_config

In [6]:
opt_params = ckpt_cfg["sde_params"]
model_config = ckpt_cfg["saved_config"]
# Load the model
seed = 42
rng_key = jax.random.PRNGKey(seed)
drift, noise, _ = load_model_from_config(
    model_config, seed = seed, class_drift=EngineModelV2, class_noise=NoiseTerm
)

In [7]:
import matplotlib
matplotlib.use("Qt5Agg")  # Or another interactive backend like "TkAgg", Qt5Agg
# matplotlib.use("widget")  # Or another interactive backend like "TkAgg", Qt5Agg
import matplotlib.pyplot as plt

In [8]:
# Some plotting utils
LABELS_TO_PLOT = drift.name_states + \
    ["inv_gear_ratio", "engine_torque", "engine_torque_est", "torque_shaft", "rear_wheel_speed_res"]
max_num_rows = 3
# Optimize for more or less equal number of rows and columns
num_rows = min(max_num_rows, len(LABELS_TO_PLOT))
num_cols = int(np.ceil(len(LABELS_TO_PLOT) / num_rows))
fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 8), sharex=True)
ax_flatten = axes.flatten()
# Create the lines for the ground truth
lines_gt = {}
for idx, name in enumerate(LABELS_TO_PLOT):
    lines_gt[name], = ax_flatten[idx].plot([], [], label="Ground Truth", color="blue", linewidth=1)
    ax_flatten[idx].set_title(name)
    ax_flatten[idx].grid()
    ax_flatten[idx].set_xlabel("Time [s]")
    ax_flatten[idx].legend()

# List to store multiple preds
pred_lines = { name : [] for name in LABELS_TO_PLOT }

fig.tight_layout()
fig.show()

def update_plots(
    times,
    gt_state,
    pred_state,
    alpha = 0.8,
    do_not_erase= False
):
    for idx, name in enumerate(LABELS_TO_PLOT):
        curr_ax = ax_flatten[idx]
        # Update the ground truth
        if not do_not_erase:
            if name in gt_state:
                lines_gt[name].set_data(times, gt_state[name])
            if name in pred_state:
                for line in pred_lines[name]:
                    line.remove()
                pred_lines[name].clear()
        elif name in gt_state:
            # We want to add te new data to the existing one
            lines_gt[name].set_data(
                np.concatenate((lines_gt[name].get_xdata(), times)),
                np.concatenate((lines_gt[name].get_ydata(), gt_state[name]))
            )
        if name not in pred_state:
            curr_ax.relim()
            curr_ax.autoscale_view()
            continue
        # Create the new prediction line
        n_trajectories = pred_state[name].shape[0]
        for i in range(n_trajectories):
            pred_line, = curr_ax.plot(
                times, pred_state[name][i], color="red", alpha=alpha, linewidth=1,
                label="Prediction" if i == 0 else "_nolegend_"
            )
            pred_lines[name].append(pred_line)
        # Refres axis and so on
        curr_ax.relim()
        curr_ax.autoscale_view()
    fig.canvas.draw()
    fig.canvas.flush_events()

QStandardPaths: wrong permissions on runtime directory /run/user/1000/, 0755 instead of 0700


In [9]:
# Problem configuration
# Problem config for data extraction
problem_config_for_sampling ={
    "names_states": list(drift.name_states) + ["time",],
    "names_controls": list(drift.name_controls),
    "time_dependent_parameters": ["inv_gear_ratio", "engine_torque"],
    "system_physical_params": {},
}
action_sampling_strategy ={
    "default": "first"
}

# Let's define function for prediction to use in plotting
@partial(jax.jit, static_argnames=["num_samples", "num_iter_between_dt"])
def prediction_paths(
    init_state,
    controls,
    times,
    rng_key,
    hist_states,
    params,
    num_samples = 4,
    num_iter_between_dt = 1
):
    # Initialize the state and control
    rng_key = jax.random.split(rng_key, num_samples)
    vmap_integrate = jax.vmap(
        integrate_path,
        in_axes=(None, None, None, 0, None, None, None, None, None, None)
    )
    # Integrate the dynamics for each sample
    paths, extras = vmap_integrate(
        init_state, controls, times, rng_key, params, hist_states, drift, noise,
        True, num_iter_between_dt
    )
    state_dict = { name : paths[..., idx] \
        for idx, name in enumerate(drift.name_states + drift.name_latents)
    }
    return {**extras, **state_dict}

def generate_examples(
    m_params, dataset, rng_key, num_samples,
    stepsize_range=(2, 2), horizon=100, num_iter_between_dt=1,
):
    # Get the batch
    test_batch_data = pick_batch_transitions_as_array(
        dataset, 1, stepsize_range, horizon, 
        problem_config_for_sampling, action_sampling_strategy
    ) # Smple a batch with a single element
    # Extract the state and control
    states, controls, gt_extra = test_batch_data
    states, controls = states[0], controls[0]
    gt_extra = { k : v[0] for k, v in gt_extra.items() }
    _time = states[:, -1]
    states = states[:, :-1]
    # Let's remove the history
    (hist_state, hist_control), (states, controls) = \
        drift.split_state_with_history(states, controls)
    _time_hist = _time[:hist_state.shape[0]]
    _time = _time[-states.shape[0]:]
    gt_extra = {k : v[hist_control.shape[0]:] for k, v in gt_extra.items()}
    hist_states = (hist_state, hist_control, _time_hist)
    # Get the initial state
    preds = prediction_paths(
        states[0], controls, _time, rng_key, hist_states, 
        m_params, num_samples, num_iter_between_dt
    )
    # Convert the groundtruth in a dict
    _time = _time[:-1]
    gt_data = {
        **gt_extra,
        **{name : states[...,:_time.shape[0],idx] for idx, name in enumerate(drift.name_states)},
    }
    pred_data = { name : np.array(v)[...,:_time.shape[0]] for name, v in preds.items() }
    gt_data = { name : np.array(v) for name, v in gt_data.items() }
    return _time, gt_data, pred_data

In [10]:
# Load the latest data
# ---
opt_params = ckpt_cfg["sde_params"]

# Load a trajectory to test the model
test_traj_indx = 0
test_trajectory = full_dataset["trajectories"][test_traj_indx]
sim_config = {
    "num_skips": 4,
    "horizon": 200,
    "num_samples": 4,
    "num_iter_between_dt": 4,
    # "num_sim_steps": 
}
_rng_key = jax.random.PRNGKey(42)

sim_steps = sim_config["num_skips"] * sim_config["horizon"]
num_sim_steps = len(test_trajectory["time"]) // sim_steps
len_simulation = num_sim_steps * sim_steps
assert len_simulation > 0, "The simulation length is zero"
# Get the names of the states and controls
name_states = drift.name_states
name_controls = drift.name_controls
# Store the result of the simulation
result_simulation = []
num_sim_steps = sim_config.get("num_sim_steps", num_sim_steps)

# Iterate through each chunk of the trajectory
past_traj = None
for _traj in range(num_sim_steps):
    # Get the current trajectory
    indx_traj = _traj * sim_steps
    curr_sim_time = test_trajectory["time"][indx_traj]
    print(f"Current simulation time: {curr_sim_time}")
    # Current trajectory to simulate
    data_steps = sim_config["num_skips"]
    horizon_pred = (sim_config["horizon"] + 1) * data_steps
    if indx_traj + horizon_pred > len_simulation:
        break
    # Get current state for the prediction
    states_evol = np.array([
        test_trajectory[_k][indx_traj:indx_traj+horizon_pred:data_steps] \
            for _k in name_states]).T
    controls_evol = np.array([
        test_trajectory[_k][indx_traj:indx_traj+horizon_pred:data_steps][:-1] \
            for _k in name_controls]).T
    time_evol = \
        test_trajectory["time"][indx_traj:indx_traj+horizon_pred:data_steps]

    _rng_key, next_rng_key = jax.random.split(_rng_key)
    if _traj == 0:
        (_hist_state, _hist_u), _ = drift.split_state_with_history(
            states_evol, controls_evol
        )
        hist_data = (_hist_state, _hist_u, time_evol[:_hist_state.shape[0]])
    else:
        hist_data = (
            past_traj[0][-(drift.history_size+1):],
            past_traj[1][-drift.history_size:],
            past_traj[2][-(drift.history_size+1):]
        )
    past_traj = (states_evol, controls_evol, time_evol)
    # Simulate the dynamics
    pred_paths = prediction_paths(
        states_evol[0], controls_evol, time_evol, next_rng_key, hist_data,
        opt_params, num_samples=sim_config["num_samples"],
        num_iter_between_dt=sim_config["num_iter_between_dt"]
    )
    # Clean for same dim
    time_evol = time_evol[:-1]
    # Convert the pred_paths to a dict and groundtruth
    pred_paths_dict = {_k : np.array(v)[..., :time_evol.shape[0]] for _k, v in pred_paths.items()}
    gt_paths_dict = {
        _k : states_evol[..., :time_evol.shape[0], indx] for indx, _k in enumerate(name_states)
    }
    # Plot the results
    update_plots(time_evol, gt_paths_dict, pred_paths_dict, alpha=0.3, do_not_erase= _traj != 0)
    

Current simulation time: 46.019502185809
Current simulation time: 50.419502394798
Current simulation time: 54.819502603787
Current simulation time: 59.219502812775
Current simulation time: 63.619503021764
Current simulation time: 68.019503230753
Current simulation time: 72.419503439742
Current simulation time: 76.81950364873
Current simulation time: 81.219503857719
Current simulation time: 85.619504066708
Current simulation time: 90.019504275697
Current simulation time: 94.419504484686
Current simulation time: 98.819504693674
Current simulation time: 103.21950490266
Current simulation time: 107.61950511165
Current simulation time: 112.01950532064
Current simulation time: 116.41950552963
Current simulation time: 120.81950573862
Current simulation time: 125.21950594761
Current simulation time: 129.6195061566
Current simulation time: 134.01950636558
Current simulation time: 138.41950657457
Current simulation time: 142.81950678356
Current simulation time: 147.21950699255
Current simulation

In [ ]:
from train_rl import get_evaluate_env_fn, make_train

In [ ]:
# Let's make sure we can load any environment.
opt_params = ckpt_cfg["sde_params"]
model_config = ckpt_cfg["saved_config"]
# Load the model
seed = 42
rng_key = jax.random.PRNGKey(seed)
# Load ModelWithTargetWheelSpeedAndRate

In [ ]:
ckpt_name = "test_10"
ckpt_cfg, path_ckpt = load_saved_data_from_checkpoint(
    TRAINING_LOGS_DIR, ckpt_name, step = -2
)
# Let's make sure we can load any environment.
opt_params = ckpt_cfg["sde_params"]
model_config = ckpt_cfg["saved_config"]
# Load the model
seed = 42
rng_key = jax.random.PRNGKey(seed)
env_drift, env_noise, _ = load_model_from_config(
    model_config, seed = seed, 
    class_drift=ModelWithTargetWheelSpeedAndRate, class_noise=NoiseTerm
)